In [0]:
from pyspark.sql import functions as F

# Calendar range and target table
start_date = "1940-01-01"
end_date = "2030-12-31"
target_table = "adb_caresync.silver.calendar"

print("=" * 80)
print("[STEP 1] Building calendar")
print(f"  Date range   : {start_date} to {end_date}")
print(f"  Target table : {target_table}")

# One row per calendar day: generate the date sequence and explode it into rows
dates_df = spark.sql(f"""
    SELECT explode(sequence(DATE'{start_date}', DATE'{end_date}', INTERVAL 1 DAY)) AS date
""")

# ISO day of week: 1 = Monday ... 7 = Sunday (weekday() returns 0 = Monday ... 6 = Sunday)
iso_dow = F.expr("weekday(date)") + 1

calendar_df = (
    dates_df
    # Keys
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    # Year
    .withColumn("year", F.year("date"))
    .withColumn("is_leap_year", ((F.col("year") % 4 == 0) & (F.col("year") % 100 != 0)) | (F.col("year") % 400 == 0))
    .withColumn("year_start_date", F.trunc("date", "year"))
    .withColumn("year_end_date", F.last_day(F.make_date(F.col("year"), F.lit(12), F.lit(1))))
    # Half year
    .withColumn("half_year", F.when(F.month("date") <= 6, 1).otherwise(2))
    .withColumn("half_year_name", F.concat(F.lit("H"), F.col("half_year")))
    # Quarter
    .withColumn("quarter", F.quarter("date"))
    .withColumn("quarter_name", F.concat(F.lit("Q"), F.col("quarter")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.col("quarter")))
    .withColumn("quarter_start_date", F.trunc("date", "quarter"))
    .withColumn("quarter_end_date", F.date_sub(F.add_months(F.col("quarter_start_date"), 3), 1))
    .withColumn("day_of_quarter", F.datediff("date", "quarter_start_date") + 1)
    # Month
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("year_month_key", F.date_format("date", "yyyyMM").cast("int"))
    .withColumn("month_start_date", F.trunc("date", "month"))
    .withColumn("month_end_date", F.last_day("date"))
    .withColumn("days_in_month", F.dayofmonth(F.last_day("date")))
    # Week (ISO 8601: weeks start on Monday, week 1 contains the first Thursday of the year)
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("iso_year", F.year(F.date_add("date", 4 - iso_dow)))
    .withColumn("iso_year_week", F.concat(F.col("iso_year"), F.lit("-W"), F.lpad(F.col("week_of_year").cast("string"), 2, "0")))
    .withColumn("week_start_date", F.date_sub("date", F.expr("weekday(date)")))
    .withColumn("week_end_date", F.date_add(F.col("week_start_date"), 6))
    .withColumn("week_of_month", F.ceil(F.dayofmonth("date") / 7).cast("int"))
    # Day
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_year", F.dayofyear("date"))
    .withColumn("day_of_week", iso_dow)
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("day_short_name", F.date_format("date", "EEE"))
    # Flags
    .withColumn("is_weekend", iso_dow >= 6)
    .withColumn("is_weekday", iso_dow <= 5)
    .withColumn("is_month_start", F.col("date") == F.col("month_start_date"))
    .withColumn("is_month_end", F.col("date") == F.col("month_end_date"))
    .withColumn("is_quarter_start", F.col("date") == F.col("quarter_start_date"))
    .withColumn("is_quarter_end", F.col("date") == F.col("quarter_end_date"))
    .withColumn("is_year_start", F.col("date") == F.col("year_start_date"))
    .withColumn("is_year_end", F.col("date") == F.col("year_end_date"))
)

# Final column order
calendar_df = calendar_df.select(
    "date_key", "date",
    "year", "is_leap_year", "year_start_date", "year_end_date",
    "half_year", "half_year_name",
    "quarter", "quarter_name", "year_quarter", "quarter_start_date", "quarter_end_date", "day_of_quarter",
    "month", "month_name", "month_short_name", "year_month", "year_month_key",
    "month_start_date", "month_end_date", "days_in_month",
    "week_of_year", "iso_year", "iso_year_week", "week_start_date", "week_end_date", "week_of_month",
    "day_of_month", "day_of_year", "day_of_week", "day_name", "day_short_name",
    "is_weekend", "is_weekday",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end", "is_year_start", "is_year_end",
)

print(f"  Columns ({len(calendar_df.columns)}) : {calendar_df.columns}")
print("=" * 80)

In [0]:
# Overwrite the calendar table (static reference data, safe to fully rebuild)
print(f"[STEP 2] Writing calendar to {target_table} (mode = overwrite)")

calendar_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

comment_sql = f"COMMENT ON TABLE {target_table} IS 'Standard calendar dimension: one row per calendar day from {start_date} to {end_date}. date_key = yyyyMMdd. Weeks follow ISO 8601 (Monday start).'"
print(f"  Query prepared:\n    {comment_sql}")
spark.sql(comment_sql)

print(f"  Write completed for {target_table}")
print("=" * 80)

In [0]:
# Validate: row count, date range, uniqueness of date_key and gaps between days
validation_sql = f"""
    SELECT
      COUNT(*)                                    AS row_count,
      COUNT(DISTINCT date_key)                    AS distinct_date_keys,
      MIN(date)                                   AS min_date,
      MAX(date)                                   AS max_date,
      datediff(MAX(date), MIN(date)) + 1          AS expected_days
    FROM {target_table}
"""
print("[STEP 3] Validating calendar table")
print(f"  Query prepared:{validation_sql}")
validation = spark.sql(validation_sql).first()
print(f"  Row count          : {validation.row_count}")
print(f"  Distinct date_keys : {validation.distinct_date_keys}")
print(f"  Date range         : {validation.min_date} to {validation.max_date}")
print(f"  Expected days      : {validation.expected_days}")

if validation.row_count == validation.distinct_date_keys == validation.expected_days:
    print("  Validation passed: one row per calendar day, no gaps or duplicates")
else:
    raise ValueError("Validation failed: calendar has duplicates or missing days")
print("=" * 80)

display(spark.table(target_table).orderBy("date").limit(10))